In [14]:

import sys
import os
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
sys.path.append(os.path.join(project_root, "models"))
from efficient_model import EfficientNetB0_Model
from resnet_model import ResNet50_Model
from mobilenet_model import MobileNetV2_Model

data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')
sys.path.append(data_path)

# 4. Importer DataLoaders depuis data_loading.py
sys.path.append(os.path.join(project_root, "data"))
from data_loading import DataLoader

import tensorflow as tf
import tensorflow.keras.backend as K
import gc

In [ ]:
print(f"Keras version: {tf.keras.__version__}")


Keras version: 3.8.0


In [10]:
print("\n" + "="*60)
print("CHARGEMENT DU DATASET SIPAKMED")
print("="*60)

DATA_PATH = os.path.join(project_root, "data", "processed") + "/"

loader = DataLoader(
        path=DATA_PATH,
        augment=True,
        batch_size=32,
        target_size=(224, 224)
    )
    
train_gen, test_gen = loader.get_generators()
    
summary = loader.get_summary()
    
print(f"\nDonnées chargées !")
print(f"Résumé:")
print(f"   Classes: {summary['classes']}")
print(f"   Nombre de classes: {summary['num_classes']}")
print(f"   Images d'entraînement: {summary['train_samples']}")
print(f"   Images de test: {summary['test_samples']}")
print(f"   Taille des batches: {summary['batch_size']}")
print(f"   Taille des images: {summary['target_size']}")
    
print(f"\nDétail par classe:")
counts = summary['counts']
for cls in summary['classes']:
    train_count = counts['train'].get(cls, 0)
    test_count = counts['test'].get(cls, 0)
    print(f"   {cls}: {train_count} train, {test_count} test")


CHARGEMENT DU DATASET SIPAKMED
Vérification des données à: c:\Users\sarah\Documents\M Info\M2 SID\AASD\Apprentisaage\data\processed/
Données déjà présentes localement
Found 551 images belonging to 3 classes.


Found 551 images belonging to 3 classes.
Found 177 images belonging to 3 classes.
DataLoader initialisé:
   - Classes: ['Abnormal', 'Benign', 'Normal']
   - Batch size: 32
   - Image size: (224, 224)
   - Augmentation: Activée

Données chargées !
Résumé:
   Classes: ['Abnormal', 'Benign', 'Normal']
   Nombre de classes: 3
   Images d'entraînement: 551
   Images de test: 177
   Taille des batches: 32
   Taille des images: (224, 224)

Détail par classe:
   Abnormal: 219 train, 80 test
   Benign: 216 train, 55 test
   Normal: 116 train, 42 test


In [11]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1),
    EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights=True),
]

In [18]:
def train_and_evaluate(model_class, model_name, **model_kwargs):

    print("\n==============================")
    print(f"  Entraînement du modèle : {model_name}")
    print("==============================\n")

    model_obj = model_class(
        input_shape=(224, 224, 3),
        num_classes=3,
        learning_rate=1e-4,
        **model_kwargs
    )
    model = model_obj.build_model() 

    # ---- PHASE 1 ----
    model.fit(
        train_gen,
        epochs=15,
        validation_data=test_gen,
        callbacks=callbacks,
        verbose=2
    )

    # ---- FINE TUNING ----
    for layer in model.layers:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        train_gen,
        epochs=10,
        validation_data=test_gen,
        callbacks=callbacks,
        verbose=2
    )

    # ================ ÉVALUATION ================
    print(f"\nÉvaluation du modèle {model_name} :")
    test_loss, test_accuracy = model.evaluate(test_gen, verbose=2)
    print(f"> Accuracy : {test_accuracy * 100:.2f}%")

    # ==== EXTRACTION LABELS ====
    y_true = []
    for i in range(len(test_gen)):
        _, label = test_gen[i]
        y_true.append(label[0])

    y_true = np.array(y_true)
    y_true_classes = np.argmax(y_true, axis=1)

    # ==== PRÉDICTIONS ====
    y_pred = model.predict(test_gen)
    y_pred_classes = np.argmax(y_pred, axis=1)

    # ==== AUC ====
    auc = roc_auc_score(y_true, y_pred, multi_class='ovr')
    print(f"> AUC : {auc:.4f}")

    # ==== RAPPORT ====
    print("\nRapport de classification :")
    print(classification_report(
        y_true_classes,
        y_pred_classes,
        target_names=loader.class_names
    ))

    # ==== MATRICE ====
    print("\nMatrice de confusion :")
    print(confusion_matrix(y_true_classes, y_pred_classes))

    return {
        "model": model_name,
        "accuracy": test_accuracy,
        "auc": auc,
        "model_object": model_obj,
        "trained_model": model
    }

In [19]:

# Trouver les classes dans les modules
ResNet50_Model = ResNet50_Model  # ou le nom exact trouvé
EfficientNetB0_Model = EfficientNetB0_Model
MobileNetV2_Model = MobileNetV2_Model
results = []

results.append(train_and_evaluate(ResNet50_Model, "ResNet50"))
K.clear_session()
gc.collect()

results.append(train_and_evaluate(EfficientNetB0_Model, "EfficientNetB0"))
K.clear_session()
gc.collect()

results.append(train_and_evaluate(MobileNetV2_Model, "MobileNetV2"))
K.clear_session()
gc.collect()



  Entraînement du modèle : ResNet50

Epoch 1/15
18/18 - 62s - 3s/step - accuracy: 0.4446 - loss: 6.0107 - val_accuracy: 0.6271 - val_loss: 5.3382 - learning_rate: 1.0000e-04
Epoch 2/15
18/18 - 52s - 3s/step - accuracy: 0.5481 - loss: 5.5539 - val_accuracy: 0.7910 - val_loss: 5.0254 - learning_rate: 1.0000e-04
Epoch 3/15
18/18 - 49s - 3s/step - accuracy: 0.6316 - loss: 5.2368 - val_accuracy: 0.8136 - val_loss: 4.8463 - learning_rate: 1.0000e-04
Epoch 4/15
18/18 - 49s - 3s/step - accuracy: 0.6860 - loss: 5.0721 - val_accuracy: 0.7740 - val_loss: 4.7156 - learning_rate: 1.0000e-04
Epoch 5/15
18/18 - 48s - 3s/step - accuracy: 0.7024 - loss: 4.9017 - val_accuracy: 0.7910 - val_loss: 4.5780 - learning_rate: 1.0000e-04
Epoch 6/15
18/18 - 49s - 3s/step - accuracy: 0.7495 - loss: 4.7142 - val_accuracy: 0.8475 - val_loss: 4.4315 - learning_rate: 1.0000e-04
Epoch 7/15
18/18 - 48s - 3s/step - accuracy: 0.7350 - loss: 4.6051 - val_accuracy: 0.8531 - val_loss: 4.3264 - learning_rate: 1.0000e-04
Epo

KeyboardInterrupt: 

In [ ]:
print("\n\n==============================")
print("     TABLEAU FINAL DES SCORES")
print("==============================\n")

for r in results:
    print(f"{r['model']}: Accuracy = {r['accuracy']:.4f} | AUC = {r['auc']:.4f}")




     TABLEAU FINAL DES SCORES

ResNet50: Accuracy = 0.9492 | AUC = 0.9960
EfficientNetB0: Accuracy = 0.9379 | AUC = 0.9934
MobileNetV2: Accuracy = 0.3107 | AUC = 0.5869
